<h1>Homework #2 - InfluxDB</h1>

<p>L’esercizio richiede di caricare in InfluxDB un dataset di dati meteo (temperature, pressioni, umidità) per diverse città (per ogni città sono riportate anche la nazione di appartenenza e le coordinate geografiche), raccolti con cadenza oraria dalle ore 12 del 01 ottobre 2012 alla mezzanotte del 30 novembre 2017. Eseguire il compito e rispondere a 5 domande analitiche utilizzando Python. Il dataset è quello disponibile a questo <a href="https://www.kaggle.com/datasets/selfishgene/historical-hourly-weather-data" target="_blank">link</a>.</p>
    
<p>Import delle librerie e delle classi necessarie allo svolgimento del compito</p>

In [ ]:
import pandas as pd
import numpy as np
import time
from datetime import datetime
from influxdb_client import InfluxDBClient, Point, WritePrecision
# from influxdb_client.client.write_api import SYNCHRONOUS
# from influxdb_client.client.write_api import ASYNCHRONOUS
from influxdb_client.client.write_api import WriteOptions
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from river import cluster, stream

# Disattivazione dei warnings derivanti dall'uso combinato di InfluxDB e Pandas
import warnings
from influxdb_client.client.warnings import MissingPivotFunction
warnings.simplefilter("ignore", MissingPivotFunction)

# =============================
# CONFIGURAZIONE INFLUXDB
# =============================
token = "mZBeBXdweGwub1_q87IjKAB2_1ofWpxn6fQyHt8Eaezr5B4cQlp-eUiUM1nQmcP8MKgusDgNPYJnE4PFM8Z4Bw=="
org = "University of Brescia"
bucket = "weather_data"
url = "http://localhost:8086"

dataset_dir = "./weather_data/"

<h2>Caricamento dei dati dentro InfluxDB</h2>

<p>Scaricare il dataset e simulare in Python il funzionamento di N sensori (uno per ogni città e per ogni grandezza misurata) per caricare i dati dentro un database InfluxDB. Il periodo di rilevazione è rappresentato dalla colonna temporale "datetime". Durante il caricamento, aggiungere un tag per specificare la stagione meteorologica (Inverno, Primavera, Estate, Autunno), basata sulla data di registrazione.</p>

<p>Durante il salvataggio, impostare delle soglie di warning e di allarme sui valori di umidità e temperatura, secondo i criteri seguenti: (a) se l’umidità è superiore al 90%, emettere un segnale di pre-allerta (warning); se invece l’umidità supera il 98%, emettere un segnale di allerta; (b) se la temperatura è superiore a 45°C, emettere un segnale di allerta; (c) escludere direttamente durante la fase di caricamento i valori di temperatura non compresi tra -50 e 50°C; (d) escludere direttamente durante la fase di caricamento i valori di umidità non compresi tra 0 e 100%.</p>

In [ ]:
simulate_realtime = False
interval_seconds = 1

# =============================
# FUNZIONI DI SUPPORTO
# =============================

def get_season(date: pd.Timestamp):
    y = date.year

    inverno_start = pd.Timestamp(f"{y}-12-21")
    primavera_start = pd.Timestamp(f"{y}-03-21")
    estate_start = pd.Timestamp(f"{y}-06-21")
    autunno_start = pd.Timestamp(f"{y}-09-21")

    if date >= inverno_start or date < primavera_start:
        return "Inverno"
    elif primavera_start <= date < estate_start:
        return "Primavera"
    elif estate_start <= date < autunno_start:
        return "Estate"
    elif autunno_start <= date < inverno_start:
        return "Autunno"

def load_all_measurements(dataset_dir):
    """Carica tutti i file CSV (temperature.csv, humidity.csv, ecc.) e li restituisce come dict."""
    measurements = {}
    files = [
        "temperature.csv",
        "humidity.csv",
        "wind_speed.csv",
        "pressure.csv",
        "wind_direction.csv",
        "weather_description.csv"
    ]
    for f in files:
        try:
            df = pd.read_csv(dataset_dir + f)
            df["datetime"] = pd.to_datetime(df["datetime"])
            measurements[f.replace(".csv", "")] = df
        except FileNotFoundError:
            print(f"⚠️ File non trovato: {f}")
    return measurements


# =============================
# CONNESSIONE A INFLUXDB
# =============================
client = InfluxDBClient(url=url, token=token, org=org)

# write_api = client.write_api(write_options=ASYNCHRONOUS)
# Scrittura asincrona con batching controllato
write_api = client.write_api(
    write_options=WriteOptions(
        batch_size=1000,        # numero di punti per batch
        flush_interval=10_000,  # flush automatico ogni 10 secondi
        jitter_interval=2_000,  # aggiunge un po' di random per evitare spike
        retry_interval=5_000,   # ritenta in caso di errore temporaneo
    )
)

# =============================
# CARICAMENTO DATI
# =============================

start_time = time.time()

# Carica tutti i dataframe
data_dict = load_all_measurements(dataset_dir)

# Carica attributi città (nation, lat, lon)
city_attr_file = dataset_dir + "city_attributes.csv"
try:
    city_attr_df = pd.read_csv(city_attr_file)
    city_attr_df["City"] = city_attr_df["City"].astype(str)
    city_to_nation = dict(zip(city_attr_df["City"], city_attr_df["Country"]))
except FileNotFoundError:
    print("⚠️ File city_attributes.csv non trovato: i tag 'nation' non saranno aggiunti.")
    city_to_nation = {}

# Ottieni la lista di città dalle colonne (escludendo datetime)
cities = [c for c in data_dict["temperature"].columns if c != "datetime"]
timestamps = data_dict["temperature"]["datetime"]

# Cache stagioni
season_cache = {ts: get_season(ts) for ts in timestamps}

print(f"🏙️ Città trovate: {len(cities)} -> {', '.join(cities)}")
print(f"🕒 Timestamp totali: {len(timestamps)}\n")

nations_used = set()

# =============================
# SCRITTURA DATI IN INFLUXDB
# =============================

for city in cities:
    nation = city_to_nation.get(city, "Unknown")
    nations_used.add(nation)
    
    print(f"⏳ Caricamento dati per {city} ({nation}) ...")
    desc_df = data_dict.get("weather_description", None)
    batch_count = 0

    for idx, row in data_dict["temperature"].iterrows():
        timestamp = row["datetime"]
        season = season_cache[timestamp]

        # Descrizione meteo (come tag)
        desc_value = "Unknown"
        if desc_df is not None and city in desc_df.columns and idx < len(desc_df):
            val = desc_df.iloc[idx][city]
            if pd.notna(val):
                desc_value = str(val)

        # --- Lettura e validazione temperatura ---
        temp_kelvin = data_dict["temperature"].at[idx, city]
        if pd.isna(temp_kelvin):
            continue

        try:
            temp_celsius = float(temp_kelvin) - 273.15
        except ValueError:
            continue

        # filtro valori assurdi
        if temp_celsius < -50 or temp_celsius > 50:
            continue

        # --- Lettura e validazione umidità ---
        humidity_df = data_dict.get("humidity")
        if humidity_df is None or city not in humidity_df.columns:
            continue

        humidity = humidity_df.at[idx, city]
        if pd.isna(humidity):
            continue

        try:
            humidity = float(humidity)
        except ValueError:
            continue

        if humidity < 0 or humidity > 100:
            continue

        # Crea il punto "weather"
        point = (
            Point("weather")
            .tag("city", city)
            .tag("nation", nation)
            .tag("season", season)
            .tag("weather_description", desc_value)
            .field("temperature", temp_celsius)
            .field("humidity", humidity)
            .time(timestamp, WritePrecision.S)
        )

        # Aggiunge le fields non filtrate (una per ciascuna misura)
        for meas in ["wind_speed", "pressure", "wind_direction"]:
            df = data_dict.get(meas)
            if df is not None and city in df.columns:
                val = df.at[idx, city]
                if pd.notna(val):
                    try:
                        point.field(meas, float(val))
                    except ValueError:
                        continue  # ignora valori non numerici

        write_api.write(bucket=bucket, org=org, record=point)
        batch_count += 1

        # Log periodico
        if idx % 2000 == 0 and idx > 0:
            print(f"  ... {idx} punti elaborati per {city}")

        # Simulazione realtime (facoltativa)
        if simulate_realtime:
            print(f"→ Sent {city} at {timestamp}")
            time.sleep(interval_seconds)

    # Flush esplicito dopo ogni città
    write_api.flush()
    print(f"✅ Completato: {city} ({batch_count} punti scritti)\n")

# =============================
# CHIUSURA
# =============================
write_api.flush()
client.close()

end_time = time.time()
print("\n🌦️ Tutti i dati validi sono stati caricati in InfluxDB con successo!")
print(f"⏱️ Tempo di esecuzione: {end_time - start_time:.2f} secondi")
print(f"🏳️ Nazioni trovate: {sorted(nations_used)}\n")

missing = set(cities) - set(city_to_nation.keys())
if missing:
    print(f"⚠️ Attenzione: nessuna nazione trovata per: {', '.join(missing)}")

<p><b>Considerazioni sull'impostazione degli alert (non gestiti nel codice mostrato qui sopra).</b> 

<table>
      <caption>Obiettivi e soluzioni per alert su dati storici in InfluxDB</caption>
      <thead>
        <tr>
          <th scope="col">Obiettivo</th>
          <th scope="col">Soluzione</th>
        </tr>
      </thead>
      <tbody>
        <tr>
          <td>Analizzare alert su dati passati</td>
          <td>Query manuale con Flux e gestione degli alert a livello applicativo (e.g., Python)</td>
        </tr>
        <tr>
          <td>Testare la logica di alert su storico</td>
          <td>Task con <code>range()</code> fisso</td>
        </tr>
        <tr>
          <td>Far “scattare” alert per dati passati</td>
          <td>Replay con timestamp aggiornato</td>
        </tr>
      </tbody>
    </table>
</p>

<p>Per impostare un task (soluzione #2), è necessario andare in <strong>Tasks → + Create Task → New Task</strong>, e inserire uno script simile a questo:</p>

  <pre><code>option task = { 
  name: "check_high_temperature_past",
  every: 1h,                  // (opzionale) esegue ogni ora
}

from(bucket: "sensor_data")
  |> range(start: 2012-10-01T12:00:00Z, stop: 2017-11-30T00:00:00Z) // periodo storico
  |> filter(fn: (r) => r._measurement == "temperature")
  |> filter(fn: (r) => r._field == "_value")
  |> filter(fn: (r) => r._value > 80.0)
  |> group(columns: ["device_id"], mode: "by")
  |> count()
  |> set(key: "_measurement", value: "alert_results")
  |> to(bucket: "alerts", org: "YourOrg")</code></pre>

<h2>Query #1</h2>
<p>Calcolare la temperatura media per ogni città, raggruppata per stagione e visualizzarla in un grafico a barre.</p>

In [ ]:
client = InfluxDBClient(url=url, token=token, org=org)
query_api = client.query_api()

# === QUERY FLUX ===
query = f"""
from(bucket: "{bucket}")
  |> range(start: 2012-10-01T12:00:00Z, stop: 2017-11-30T00:00:00Z)
  |> filter(fn: (r) => r["_measurement"] == "weather")
  |> filter(fn: (r) => r["_field"] == "temperature")
  |> group(columns: ["city", "season"])
  |> mean()
  |> keep(columns: ["_value", "city", "season"])
"""

# === ESECUZIONE QUERY ===
df = query_api.query_data_frame(query)
client.close()

# InfluxDB può restituire una lista di DataFrame → concatenali
if isinstance(df, list):
    df = pd.concat(df)

# === PULIZIA E RINOMINA ===
df = df.rename(columns={"_value": "temperature_mean"})

# === VERIFICA DATI ===
print("Esempio dati ottenuti:")
print(df.head())
print("\nCittà uniche:", df["city"].unique())
print("Stagioni presenti:", df["season"].unique())

# === PREPARAZIONE PER IL GRAFICO ===
pivot_df = df.pivot(index="city", columns="season", values="temperature_mean")

# Ordina le stagioni in ordine naturale
order = ["Inverno", "Primavera", "Estate", "Autunno"]
pivot_df = pivot_df[[s for s in order if s in pivot_df.columns]]

# === GRAFICO ===
plt.figure(figsize=(12, 6))
pivot_df.plot(kind="bar", figsize=(12, 6))
plt.title("Temperatura media stagionale per città (2012–2017)", fontsize=14)
plt.xlabel("Città")
plt.ylabel("Temperatura media (°C)")
plt.legend(title="Stagione")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

<h2>Query #2</h2>
<p>Calcolare la variazione (massimo - minimo) di umidità giornaliera per ogni città, per un mese a scelta, e visualizzarla in un grafico a linee.</p>

In [ ]:
client = InfluxDBClient(url=url, token=token, org=org)
query_api = client.query_api()

month_start = "2016-06-01T00:00:00Z"
month_end   = "2016-06-30T23:59:59Z"

query = f"""
from(bucket: "{bucket}")
  |> range(start: {month_start}, stop: {month_end})
  |> filter(fn: (r) => r["_measurement"] == "weather")
  |> filter(fn: (r) => r["_field"] == "humidity")
  |> aggregateWindow(every: 1d, fn: spread, createEmpty: false)
  |> group(columns: ["city"])
  |> keep(columns: ["_time", "city", "_value"])
"""

df = query_api.query_data_frame(query)
client.close()

df["_time"] = pd.to_datetime(df["_time"])
df = df.sort_values(["city", "_time"])
df.rename(columns={"_value": "humidity_spread"}, inplace=True)

for city, data in df.groupby("city"):
    plt.figure(figsize=(10, 4))
    plt.plot(data["_time"], data["humidity_spread"], marker="o")
    plt.title(city)
    plt.xlabel("Data")
    plt.ylabel("Δ Umidità (%)")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

<h2>Query #3</h2>
<p>Trovare le giornate con il picco massimo di precipitazioni per ogni città nel corso dell'ultimo anno disponibile.</p>

In [ ]:
client = InfluxDBClient(url=url, token=token, org=org)
query_api = client.query_api()

# Ultimo anno disponibile (puoi modificarlo se vuoi renderlo dinamico)
year_start = "2016-01-01T00:00:00Z"
year_end   = "2016-12-31T23:59:59Z"

query = f"""
from(bucket: "{bucket}")
  |> range(start: {year_start}, stop: {year_end})
  |> filter(fn: (r) => r["_measurement"] == "weather")
  |> filter(fn: (r) => r["_field"] == "humidity")
  |> aggregateWindow(every: 1d, fn: max, createEmpty: false)
  |> group(columns: ["city"])
  |> keep(columns: ["_time", "city", "_value"])
"""

df = query_api.query_data_frame(query)
client.close()

# --- Preparazione ---
df["_time"] = pd.to_datetime(df["_time"])
df.rename(columns={"_value": "daily_max_humidity"}, inplace=True)

# Trova per ogni città il giorno con il massimo dell'anno
max_days = df.loc[df.groupby("city")["daily_max_humidity"].idxmax()]

# Mostra a video i risultati
print("📍 Giorno con massima umidità per ogni città (ultimo anno):\n")
for _, row in max_days.iterrows():
    print(f"{row['city']}: {row['_time'].date()} → {row['daily_max_humidity']:.1f}%")

<h2>Query #4</h2>
<p>Calcolare la temperatura media mensile per ciascuna città, creare un grafico che mostra l'andamento delle temperature medie nel tempo e indicare le città con i trend più marcati (in crescita o in calo).</p>

In [ ]:
start = "2012-10-01T12:00:00Z"
stop  = "2017-11-30T00:00:00Z"

client = InfluxDBClient(url=url, token=token, org=org)
query_api = client.query_api()

# -------------------------
# QUERY FLUX: media mensile per città
# -------------------------
flux = f'''
from(bucket: "{bucket}")
  |> range(start: {start}, stop: {stop})
  |> filter(fn: (r) => r["_measurement"] == "weather")
  |> filter(fn: (r) => r["_field"] == "temperature")
  |> keep(columns: ["_time", "_value", "city"])
  |> group(columns: ["city"])
  |> aggregateWindow(every: 1mo, fn: mean, createEmpty: false)
'''

# esegui query
df = query_api.query_data_frame(flux, org=org)

# concatena se Influx restituisce lista di df
if isinstance(df, list):
    df = pd.concat(df, ignore_index=True)

client.close()

# -------------------------
# PREPARAZIONE DATAFRAME
# -------------------------
df = df.rename(columns={"_time": "time", "_value": "temperature"})
df["time"] = pd.to_datetime(df["time"])
df = df.dropna(subset=["temperature", "city"])
df = df.set_index("time")

# Pivot: righe = mesi, colonne = città
pivot = df.pivot_table(index="time", columns="city", values="temperature")

# -------------------------
# CALCOLO TREND LINEARE PER CITTÀ
# -------------------------
slopes = {}
x_index = np.arange(len(pivot.index))
for city in pivot.columns:
    y = pivot[city].values
    mask = ~np.isnan(y)
    if mask.sum() >= 2:
        x = x_index[mask]
        yy = y[mask]
        p = np.polyfit(x, yy, 1)
        slopes[city] = p[0]*12  # °C / anno
    else:
        slopes[city] = np.nan

slopes_df = pd.DataFrame.from_dict(slopes, orient="index", columns=["slope_year"])
slopes_df = slopes_df.sort_values("slope_year", ascending=False)

top_inc = slopes_df.head(3)
top_dec = slopes_df.tail(3)

print("Top 3 città in crescita (°C/anno):")
print(top_inc.round(3))
print("\nTop 3 città in calo (°C/anno):")
print(top_dec.round(3))

# -------------------------
# PLOT
# -------------------------
plt.figure(figsize=(14, 7))

# tutte in grigio
for city in pivot.columns:
    plt.plot(pivot.index, pivot[city], color="lightgray", linewidth=0.8, zorder=1)

# evidenzia top increasing e decreasing
colors_up = ["tab:red", "tab:orange", "tab:purple"]
colors_down = ["tab:blue", "tab:cyan", "tab:green"]

for i, city in enumerate(top_inc.index):
    plt.plot(pivot.index, pivot[city], color=colors_up[i%3], linewidth=2.5, label=f"{city} (↑ {slopes[city]:.2f}°K/yr)")

for i, city in enumerate(top_dec.index[::-1]):
    plt.plot(pivot.index, pivot[city], color=colors_down[i%3], linewidth=2.5, linestyle="--", label=f"{city} (↓ {slopes[city]:.2f}°C/yr)")

plt.title("Andamento temperatura media mensile per città (trend)")
plt.xlabel("Mese")
plt.ylabel("Temperatura media (°C)")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize="small")
plt.tight_layout()
plt.show()

<h2>Query #5</h2>
<p>Calcolare per ogni città la media annuale dell’umidità e trovare quelle con i valori medi più alti e più bassi.</p>

In [ ]:
# Aumentiamo il timeout a 60 secondi
client = InfluxDBClient(url=url, token=token, org=org, timeout=60_000)
query_api = client.query_api()

# === QUERY FLUX OTTIMIZZATA ===
# - Riduciamo il range per evitare grandi volumi di dati
# - Aggregate mensile prima dell'annuale per ridurre i punti
flux = f'''
from(bucket: "{bucket}")
  |> range(start: 2012-01-01T00:00:00Z, stop: 2017-12-31T23:59:59Z)
  |> filter(fn: (r) => r["_measurement"] == "weather")
  |> filter(fn: (r) => r["_field"] == "humidity")
  |> group(columns: ["city"])
  |> aggregateWindow(every: 1y, fn: mean, createEmpty: false)
'''

# === ESECUZIONE QUERY ===
df = query_api.query_data_frame(flux, org=org)
client.close()

# Influx a volte restituisce una lista di DataFrame → concatenali
if isinstance(df, list):
    df = pd.concat(df)

# Rinomina colonna
df = df.rename(columns={"_value": "humidity_mean"})

# Calcola città con media più alta e più bassa
agg = df.groupby("city")["humidity_mean"].mean()

# Ordina in modo decrescente
agg_sorted = agg.sort_values(ascending=False)

# Prime 5 e ultime 5 città
top5 = agg_sorted.head(5)
bottom5 = agg_sorted.tail(5)

print("Prime 5 città con umidità media più alta:")
print(top5)
print("\nUltime 5 città con umidità media più bassa:")
print(bottom5)

# === GRAFICO ===
plt.figure(figsize=(12,6))
agg.sort_values(ascending=False).plot(kind="bar", color="skyblue")
plt.title("Umidità media annuale per città (2012–2017)")
plt.xlabel("Città")
plt.ylabel("Umidità media (%)")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

<h2>Consigli per migliorare l’efficienza</h2>
<ul>
<li>Ridurre il range temporale se possibile, o usare finestre di aggregazione (<code>aggregateWindow</code>) più grandi (1 mese o 1 anno invece di 1 giorno).</li>
<li>Usare <code>group()</code> mirato solo sulle colonne necessarie (<code>city</code> e <code>season</code>) per ridurre il lavoro di InfluxDB.</li>
<li>Selezionare solo i campi necessari (<code>_field</code> specifico), invece di leggere tutti i dati.</li>
<li>Materializzare i dati aggregati in un bucket separato per query frequenti (es. temperature/umidità medie stagionali già calcolate).</li>
<li>Query parallele: per 4 stagioni potresti fare 4 query parallele con Python <code>concurrent.futures.ThreadPoolExecutor</code>.</li>
</ul>

<h2>Clustering incrementale</h2>

<p>Applicando un algoritmo di clustering incrementale, verificare se in ogni nazione esistono dei valori di temperatura attorno a dei valori ricorrenti, differenziati per stagione, e se questi valori sono cambiati dal 2012 al 2017.</p>

In [ ]:
def query_sensor_data(client, bucket_name, start_time, end_time):
    query = f'''
    from(bucket: "{bucket_name}") 
      |> range(start: {start_time}, stop: {end_time}) 
      |> filter(fn: (r) => r._measurement == "sensore_iot")
      |> keep(columns: ["_time", "_value", "_field"])
    '''
    
    try:
        result = client.query_api().query(query)
        data = []
        for table in result:
            for record in table.records:
                data.append((record.get_time(), record.get_value(), record.get_field()))

        return pd.DataFrame(data, columns=["timestamp", "value", "field"])
    
    except Exception as e:
        print(f"Errore durante l'esecuzione della query: {e}")
        return pd.DataFrame()

def online_incremental_clustering(sensor_data, clustream_model, field_mapping):
    """
    Aggiorna CluStream con dati incrementali provenienti da un DataFrame in formato long.

    Parametri:
    - sensor_data: DataFrame con almeno colonne ['time', 'field', 'value']
    - clustream_model: modello CluStream già inizializzato
    - field_mapping: dict {nome_field_db : nome_feature_modello}

    Esempio field_mapping:
        {"temperatura": "temp", "umidita": "hum"}
    """

    # Uniforma nome della colonna timestamp
    if 'timestamp' in sensor_data.columns:
        sensor_data = sensor_data.rename(columns={'timestamp': 'time'})

    # Controllo colonne richieste
    required_cols = {'time', 'field', 'value'}
    if not required_cols.issubset(sensor_data.columns):
        raise ValueError(f"❌ Mancano colonne richieste: {required_cols - set(sensor_data.columns)}")

    # Filtra solo i campi di interesse
    subset = sensor_data[sensor_data['field'].isin(field_mapping.keys())]

    if subset.empty:
        return clustream_model

    # Pivot per creare riga con tutte le feature allineate per timestamp
    df_pivot = subset.pivot_table(index='time', columns='field', values='value').rename(columns=field_mapping)

    # Rimuove record incompleti
    df_pivot = df_pivot.dropna()

    # Streaming sul modello
    data_stream = stream.iter_pandas(df_pivot, y=None)
    for x, _ in data_stream:
        clustream_model.learn_one(x)

    return clustream_model

def offline_incremental_clustering(clustream_model, n_macro_clusters=3):
    """
    Estrae i micro-cluster dal modello CluStream, esegue il clustering offline (macro-clustering)
    e ritorna un DataFrame con i centri e le relative etichette.

    Ritorna:
    - DataFrame con colonne feature + 'macro_cluster'
    - Oggetto KMeans per eventuale uso successivo
    """

    micro_clusters = clustream_model.micro_clusters

    if not micro_clusters:
        print("⚠️ Nessun micro-cluster disponibile.")
        return pd.DataFrame(), None

    # Estrai i centri dei micro-cluster in forma generica
    if isinstance(micro_clusters, dict):
        centers = [mc.center if hasattr(mc, "center") else mc for mc in micro_clusters.values()]
    else:
        centers = [mc.center if hasattr(mc, "center") else mc for mc in micro_clusters]

    df_centers = pd.DataFrame(centers)

    # Applica il macro-clustering (offline)
    k = min(n_macro_clusters, len(df_centers))
    model = KMeans(n_clusters=k)
    df_centers['macro_cluster'] = model.fit_predict(df_centers)

    return df_centers, model

def plot_macro_clusters(df_centers, kmeans_model):
    """
    Visualizza il risultato del macro-clustering dei micro-cluster.
    Supporta n-dimensioni, ma il plot usa solo le prime 2 feature.
    """

    if df_centers.empty:
        print("⚠️ Nessun dato da visualizzare.")
        return

    feature_cols = df_centers.columns[:-1]  # tutte tranne 'macro_cluster'
    if len(feature_cols) < 2:
        print("⚠️ Servono almeno 2 feature per la visualizzazione.")
        return

    x_col, y_col = feature_cols[:2]

    plt.figure(figsize=(10, 6))

    for cluster_id in df_centers['macro_cluster'].unique():
        cluster_points = df_centers[df_centers['macro_cluster'] == cluster_id]
        plt.scatter(cluster_points[x_col], cluster_points[y_col], label=f'Macro-cluster {cluster_id}', s=150)

    # Plot dei centri macro (solo sulle prime 2 dimensioni)
    centers = kmeans_model.cluster_centers_
    plt.scatter(centers[:, 0], centers[:, 1], marker='X', s=300, c='black', label='Centri Macro')

    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title('Macro-clustering dei micro-cluster (offline)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

In [ ]:
# ----------------------------------------------------------
# 1. QUERY DATI INFLUXDB
# ----------------------------------------------------------

client = InfluxDBClient(url=url, token=token, org=org, timeout=60_000)
query_api = client.query_api()

flux = f'''
from(bucket: "{bucket}")
  |> range(start: 2012-01-01T00:00:00Z, stop: 2017-12-31T23:59:59Z)
  |> filter(fn: (r) => r["_measurement"] == "weather")
  |> filter(fn: (r) => r["_field"] == "temperature")
  |> keep(columns: ["_time", "_value", "_field", "city", "season", "nation"])
'''

df = query_api.query_data_frame(flux)
if isinstance(df, list):
    df = pd.concat(df)

df = df.rename(columns={"_time": "time", "_value": "value", "_field": "field"})
df["time"] = pd.to_datetime(df["time"])
df = df.dropna(subset=["nation"]) # per sicurezza

# ----------------------------------------------------------
# 2. CLUSTERING INCREMENTALE CON LE FUNZIONI RIUSABILI
# ----------------------------------------------------------

results = {}  # (nation, season) -> {year: centroids}
n_clusters = 3
time_window = 1000
time_gap = 10

field_mapping = {"temperature": "temp"}  # campo DB → feature modello

for nation in sorted(df["nation"].unique()):
    for season in sorted(df["season"].unique()):
        subset = df[(df["nation"] == nation) & (df["season"] == season)]
        if subset.empty:
            continue

        yearly_centroids = {}

        for year in range(2012, 2018):
            df_year = subset[subset["time"].dt.year == year]
            if len(df_year) < n_clusters:
                continue

            # Inizializza CluStream
            clustream = cluster.CluStream(
                n_macro_clusters=n_clusters,
                time_window=time_window,
                time_gap=time_gap,
                seed=42
            )

            # STREAMING INCREMENTALE (usa funzione)
            clustream = online_incremental_clustering(df_year.rename(columns={"temperature": "value", "time": "timestamp"}), 
                                                      clustream, 
                                                      field_mapping)

            # MACRO-CLUSTERING OFFLINE (usa funzione)
            df_centers, kmeans_model = offline_incremental_clustering(clustream, n_macro_clusters=n_clusters)

            if not df_centers.empty:
                centroids = sorted(df_centers.iloc[:, 0].values)  # prima colonna = "temp"
                yearly_centroids[year] = centroids

        if yearly_centroids:
            results[(nation, season)] = yearly_centroids

# ----------------------------------------------------------
# 3. VISUALIZZAZIONE TREND PER NAZIONE E STAGIONE
# ----------------------------------------------------------

for (nation, season), yearly_centroids in results.items():
    plt.figure(figsize=(8, 5))
    for i in range(n_clusters):
        x = list(yearly_centroids.keys())
        y = [centroids[i] for centroids in yearly_centroids.values()]
        plt.plot(x, y, marker="o", label=f"Cluster {i+1}")

    plt.title(f"Andamento dei cluster di temperatura — {nation} ({season})")
    plt.xlabel("Anno")
    plt.ylabel("Temperatura (°C)")
    plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()